In [37]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 读取数据 - 使用原始字符串（推荐方法）
df = pd.read_csv(r'C:\Users\dp239\OneDrive\Desktop\金融信用评分和风险预测\cs-training.csv', index_col=0)  # 第一列是序号，设为索引
print(df.head())
print(df.info())
print(df.describe())



   SeriousDlqin2yrs  RevolvingUtilizationOfUnsecuredLines  age  \
1                 1                              0.766127   45   
2                 0                              0.957151   40   
3                 0                              0.658180   38   
4                 0                              0.233810   30   
5                 0                              0.907239   49   

   NumberOfTime30-59DaysPastDueNotWorse  DebtRatio  MonthlyIncome  \
1                                     2   0.802982         9120.0   
2                                     0   0.121876         2600.0   
3                                     1   0.085113         3042.0   
4                                     0   0.036050         3300.0   
5                                     1   0.024926        63588.0   

   NumberOfOpenCreditLinesAndLoans  NumberOfTimes90DaysLate  \
1                               13                        0   
2                                4                        0   

In [38]:
# 月收入缺失：用中位数填充
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())

# 家属数缺失：用0填充（假设无家属）
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(0)

# 检查是否还有缺失
print("缺失值总数:", df.isnull().sum().sum())  # 应该是0

缺失值总数: 0


In [39]:
# ========== 1. 处理缺失值 ==========
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(0)

# ========== 2. 处理明显不合理的异常值 ==========
# 年龄：0~100之间（正常成年人）
df.loc[(df['age'] < 18) | (df['age'] > 100), 'age'] = df['age'].median()

# 额度使用率：应该在0~1之间
df.loc[df['RevolvingUtilizationOfUnsecuredLines'] > 1, 'RevolvingUtilizationOfUnsecuredLines'] = 1
df.loc[df['RevolvingUtilizationOfUnsecuredLines'] < 0, 'RevolvingUtilizationOfUnsecuredLines'] = 0

# 负债比率：应该在0~1之间
df.loc[df['DebtRatio'] > 1, 'DebtRatio'] = 1
df.loc[df['DebtRatio'] < 0, 'DebtRatio'] = 0

# 月收入：极端值截断（99%分位数）
cap_income = df['MonthlyIncome'].quantile(0.99)
df['MonthlyIncome'] = df['MonthlyIncome'].clip(upper=cap_income)

# 未结清信贷账户数量：截断99%分位数
cap_open = df['NumberOfOpenCreditLinesAndLoans'].quantile(0.99)
df['NumberOfOpenCreditLinesAndLoans'] = df['NumberOfOpenCreditLinesAndLoans'].clip(upper=cap_open)

# 房地产贷款数量：截断99%分位数
cap_real_estate = df['NumberRealEstateLoansOrLines'].quantile(0.99)
df['NumberRealEstateLoansOrLines'] = df['NumberRealEstateLoansOrLines'].clip(upper=cap_real_estate)

# 家属数量：截断99%分位数
cap_depend = df['NumberOfDependents'].quantile(0.99)
df['NumberOfDependents'] = df['NumberOfDependents'].clip(upper=cap_depend)

# 逾期次数（三个字段）：截断99%分位数
for col in ['NumberOfTime30-59DaysPastDueNotWorse', 
            'NumberOfTimes90DaysLate', 
            'NumberOfTime60-89DaysPastDueNotWorse']:
    cap = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=cap)

# ========== 3. 输出验证结果 ==========
print("处理后的最大值：")
print(df[['NumberOfOpenCreditLinesAndLoans', 
         'NumberRealEstateLoansOrLines', 
         'NumberOfDependents']].max())
print("\n处理后的 describe：")
print(df.describe())

处理后的最大值：
NumberOfOpenCreditLinesAndLoans    24.0
NumberRealEstateLoansOrLines        4.0
NumberOfDependents                  4.0
dtype: float64

处理后的 describe：
       SeriousDlqin2yrs  RevolvingUtilizationOfUnsecuredLines            age  \
count     150000.000000                         150000.000000  150000.000000   
mean           0.066840                              0.319196      52.291073   
std            0.249746                              0.349481      14.763474   
min            0.000000                              0.000000      21.000000   
25%            0.000000                              0.029867      41.000000   
50%            0.000000                              0.154181      52.000000   
75%            0.000000                              0.559046      63.000000   
max            1.000000                              1.000000      99.000000   

       NumberOfTime30-59DaysPastDueNotWorse      DebtRatio  MonthlyIncome  \
count                         150000.00000

In [40]:
import warnings
warnings.filterwarnings('ignore')
import scorecardpy as sc
import pandas as pd

# 特征列表
features = [
    'RevolvingUtilizationOfUnsecuredLines',
    'age',
    'NumberOfTime30-59DaysPastDueNotWorse',
    'DebtRatio',
    'MonthlyIncome',
    'NumberOfOpenCreditLinesAndLoans',
    'NumberOfTimes90DaysLate',
    'NumberRealEstateLoansOrLines',
    'NumberOfTime60-89DaysPastDueNotWorse',
    'NumberOfDependents'
]

# 分箱
print("开始分箱...")
bins = sc.woebin(
    df, 
    y='SeriousDlqin2yrs', 
    x=features,
    bin_num_limit=5,
    method='tree',
    positive='bad|1'
)
print("分箱完成！")

# 验证：提取 IV 并打印
iv_list = []
for var, df_bin in bins.items():
    iv = df_bin['total_iv'].iloc[0]
    iv_list.append({'variable': var, 'total_iv': iv})

iv_df = pd.DataFrame(iv_list).sort_values('total_iv', ascending=False)
print("\n各变量 IV 值：")
print(iv_df)


开始分箱...
[INFO] creating woe binning ...
分箱完成！

各变量 IV 值：
                               variable  total_iv
0  RevolvingUtilizationOfUnsecuredLines  1.078437
6               NumberOfTimes90DaysLate  0.837551
2  NumberOfTime30-59DaysPastDueNotWorse  0.740481
8  NumberOfTime60-89DaysPastDueNotWorse  0.572373
1                                   age  0.247090
5       NumberOfOpenCreditLinesAndLoans  0.075373
4                         MonthlyIncome  0.069052
7          NumberRealEstateLoansOrLines  0.055354
3                             DebtRatio  0.043943
9                    NumberOfDependents  0.033818


In [41]:
for var in features:
    print(f"\n===== {var} =====")
    df_bin = bins[var]
    print(df_bin.to_string())  # 打印完整表格


===== RevolvingUtilizationOfUnsecuredLines =====
                               variable          bin  count  count_distr   good   bad   badprob       woe    bin_iv  total_iv breaks  is_special_values
0  RevolvingUtilizationOfUnsecuredLines  [-inf,0.14)  72559     0.483727  71200  1359  0.018730 -1.322469  0.493437  1.078437   0.14              False
1  RevolvingUtilizationOfUnsecuredLines   [0.14,0.5)  36153     0.241020  34525  1628  0.045031 -0.418056  0.035232  1.078437    0.5              False
2  RevolvingUtilizationOfUnsecuredLines   [0.5,0.86)  19081     0.127207  16909  2172  0.113831  0.584077  0.055975  1.078437   0.86              False
3  RevolvingUtilizationOfUnsecuredLines   [0.86,inf)  22207     0.148047  17340  4867  0.219165  1.365737  0.493793  1.078437    inf              False

===== age =====
  variable          bin  count  count_distr   good   bad   badprob       woe    bin_iv  total_iv breaks  is_special_values
0      age  [-inf,44.0)  44508     0.296720  40015

In [42]:
import scorecardpy as sc
import pandas as pd

# 1. 手动指定负债比率的切点（合并最后两箱为 [0.52, inf)）
breaks_manual = {
    'DebtRatio': [0.02, 0.42, 0.52]   # 自动添加 inf
}

# 2. 仅对 DebtRatio 重新分箱
bins_debt = sc.woebin(df, y='SeriousDlqin2yrs', x=['DebtRatio'], 
                      breaks_list=breaks_manual, positive='bad|1')

# 3. 复制原始 bins 并替换 DebtRatio 的分箱
bins_final = bins.copy()
bins_final['DebtRatio'] = bins_debt['DebtRatio']

# 4. 查看调整后的分箱结果（使用正确的列名）
# 先获取实际的违约率列名（scorecardpy 不同版本可能不同）
bad_rate_col = 'badprob' if 'badprob' in bins_final['DebtRatio'].columns else 'bad_rate'
print("调整后的 DebtRatio 分箱（含违约率和IV）：")
print(bins_final['DebtRatio'][['bin', 'count_distr', bad_rate_col, 'total_iv']])

# 5. （可选）对比调整前后的 IV 值
print(f"\n调整前 DebtRatio IV = {bins['DebtRatio']['total_iv'].iloc[0]:.6f}")
print(f"调整后 DebtRatio IV = {bins_final['DebtRatio']['total_iv'].iloc[0]:.6f}")

[INFO] creating woe binning ...
调整后的 DebtRatio 分箱（含违约率和IV）：
           bin  count_distr   badprob  total_iv
0  [-inf,0.02)     0.088180  0.051108  0.022085
1  [0.02,0.42)     0.469193  0.060501  0.022085
2  [0.42,0.52)     0.081213  0.072894  0.022085
3   [0.52,inf)     0.361413  0.077547  0.022085

调整前 DebtRatio IV = 0.043943
调整后 DebtRatio IV = 0.022085


In [43]:
import scorecardpy as sc

# 假设你的原始数据框叫 df，分箱结果叫 bins_final（已包含调整后的负债比率分箱）
# 进行 WOE 转换，生成新数据框 df_woe，所有原始变量被替换为其 WOE 值
df_woe = sc.woebin_ply(df, bins_final)

# 查看转换后的前几行（每个变量后面加了 _woe 后缀）
print("WOE转换后的数据样例：")
print(df_woe.head())

[INFO] converting into woe values ...
WOE转换后的数据样例：
   SeriousDlqin2yrs   age_woe  NumberOfTimes90DaysLate_woe  DebtRatio_woe  \
1                 1  0.100369                    -0.389724       0.160128   
2                 0  0.449541                    -0.389724      -0.106412   
3                 0  0.449541                     2.298734      -0.106412   
4                 0  0.449541                    -0.389724      -0.106412   
5                 0  0.100369                    -0.389724      -0.106412   

   NumberRealEstateLoansOrLines_woe  MonthlyIncome_woe  \
1                          0.253629          -0.271599   
2                          0.235970           0.292569   
3                          0.235970           0.292569   
4                          0.235970           0.292569   
5                         -0.256641          -0.459905   

   NumberOfTime60-89DaysPastDueNotWorse_woe  \
1                                 -0.288208   
2                                 -0.288208

In [44]:
from sklearn.linear_model import LogisticRegression
import pandas as pd

# 特征和标签
features_woe = [col for col in df_woe.columns if col.endswith('_woe')]
X = df_woe[features_woe]
y = df_woe['SeriousDlqin2yrs']

# 训练模型（无正则化，solver='lbfgs' 支持 penalty=None）
lr = LogisticRegression(penalty=None, solver='lbfgs', max_iter=1000)
lr.fit(X, y)

# 查看系数
coef_df = pd.DataFrame({
    'variable': features_woe,
    'coefficient': lr.coef_[0]
}).sort_values('coefficient', ascending=False)
print("模型系数：")
print(coef_df)

模型系数：
                                   variable  coefficient
2                             DebtRatio_woe     1.401875
3          NumberRealEstateLoansOrLines_woe     0.615137
7  RevolvingUtilizationOfUnsecuredLines_woe     0.610412
8  NumberOfTime30-59DaysPastDueNotWorse_woe     0.535619
1               NumberOfTimes90DaysLate_woe     0.527552
9                    NumberOfDependents_woe     0.447435
0                                   age_woe     0.422520
5  NumberOfTime60-89DaysPastDueNotWorse_woe     0.402671
4                         MonthlyIncome_woe     0.283358
6       NumberOfOpenCreditLinesAndLoans_woe     0.077765


In [45]:
# 生成评分卡（使用调整后的分箱 bins_final）
card = sc.scorecard(bins_final, lr, features_woe)

# 查看评分卡所有键名（确认哪些变量）
print("评分卡包含的变量：", list(card.keys()))

# 假设键名是原始变量名（不带_woe），查看负债比率
if 'DebtRatio' in card:
    print("\n负债比率评分卡：")
    print(card['DebtRatio'])
else:
    # 如果键名带_woe
    print("\n负债比率评分卡：")
    print(card['DebtRatio_woe'])

# 对全量数据打分
df_score = sc.scorecard_ply(df, card, only_total_score=True)
print("\n前5个样本的信用评分：")
print(df_score.head())

评分卡包含的变量： ['basepoints', 'age', 'NumberOfTimes90DaysLate', 'DebtRatio', 'NumberRealEstateLoansOrLines', 'MonthlyIncome', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfOpenCreditLinesAndLoans', 'RevolvingUtilizationOfUnsecuredLines', 'NumberOfTime30-59DaysPastDueNotWorse', 'NumberOfDependents']

负债比率评分卡：
     variable          bin  points
11  DebtRatio  [-inf,0.02)    29.0
12  DebtRatio  [0.02,0.42)    11.0
13  DebtRatio  [0.42,0.52)    -9.0
14  DebtRatio   [0.52,inf)   -16.0

前5个样本的信用评分：
   score
1  470.0
2  539.0
3  418.0
4  625.0
5  538.0


In [46]:
print("基础分：", card['basepoints'])
for var in card.keys():
    if var != 'basepoints':
        print(f"\n{var} 评分卡：")
        print(card[var])

基础分：      variable  bin  points
0  basepoints  NaN   576.0

age 评分卡：
  variable          bin  points
4      age  [-inf,44.0)   -14.0
5      age  [44.0,58.0)    -3.0
6      age  [58.0,68.0)    15.0
7      age   [68.0,inf)    35.0

NumberOfTimes90DaysLate 评分卡：
                   variable         bin  points
22  NumberOfTimes90DaysLate  [-inf,1.0)    15.0
23  NumberOfTimes90DaysLate   [1.0,inf)   -87.0

DebtRatio 评分卡：
     variable          bin  points
11  DebtRatio  [-inf,0.02)    29.0
12  DebtRatio  [0.02,0.42)    11.0
13  DebtRatio  [0.42,0.52)    -9.0
14  DebtRatio   [0.52,inf)   -16.0

NumberRealEstateLoansOrLines 评分卡：
                        variable         bin  points
24  NumberRealEstateLoansOrLines  [-inf,1.0)   -10.0
25  NumberRealEstateLoansOrLines   [1.0,2.0)    11.0
26  NumberRealEstateLoansOrLines   [2.0,3.0)     8.0
27  NumberRealEstateLoansOrLines   [3.0,inf)   -11.0

MonthlyIncome 评分卡：
         variable               bin  points
15  MonthlyIncome     [-inf,5000.0)    -6.

In [47]:
df_score['score'].describe()

count    150000.000000
mean        627.430787
std          85.458537
min         235.000000
25%         582.000000
50%         649.000000
75%         690.000000
max         768.000000
Name: score, dtype: float64

In [48]:
# 分离好坏客户的分数
good_scores = df_score.loc[y == 0, 'score']
bad_scores = df_score.loc[y == 1, 'score']
print("好客户分数分布：", good_scores.describe())
print("坏客户分数分布：", bad_scores.describe())

好客户分数分布： count    139974.000000
mean        636.735629
std          75.353446
min         235.000000
25%         595.000000
50%         655.000000
75%         693.000000
max         768.000000
Name: score, dtype: float64
坏客户分数分布： count    10026.000000
mean       497.524935
std        109.119050
min        239.000000
25%        420.000000
50%        503.000000
75%        578.000000
max        760.000000
Name: score, dtype: float64


In [49]:
from sklearn.metrics import roc_auc_score, roc_curve
y_pred_prob = lr.predict_proba(X)[:, 1]
auc = roc_auc_score(y, y_pred_prob)
print(f"AUC = {auc:.4f}")

fpr, tpr, _ = roc_curve(y, y_pred_prob)
ks = max(tpr - fpr)
print(f"KS = {ks:.4f}")

AUC = 0.8548
KS = 0.5522


In [50]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
import scorecardpy as sc

# 1. 对全部数据做 WOE 转换（你已经有了 df, bins_final, features）
df_woe_full = sc.woebin_ply(df, bins_final)

# 2. 准备特征和标签
X_all = df_woe_full[[f'{f}_woe' for f in features]]
y_all = df['SeriousDlqin2yrs']

# 3. 划分训练集 (70%) 和验证集 (30%)
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.3, random_state=42
)

# 4. 训练逻辑回归（无正则化）
lr_val = LogisticRegression(penalty=None, solver='lbfgs')
lr_val.fit(X_train, y_train)

# 5. 验证集预测
y_pred_val = lr_val.predict_proba(X_val)[:, 1]

# 6. 计算指标
auc_val = roc_auc_score(y_val, y_pred_val)
print(f"验证集 AUC = {auc_val:.4f}")

fpr, tpr, _ = roc_curve(y_val, y_pred_val)
ks_val = max(tpr - fpr)
print(f"验证集 KS = {ks_val:.4f}")

[INFO] converting into woe values ...
验证集 AUC = 0.8552
验证集 KS = 0.5597


In [56]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
import scorecardpy as sc

# 1. 原始分箱（未手动调整负债比率）
bins_original = sc.woebin(
    df, 
    y='SeriousDlqin2yrs', 
    x=features, 
    bin_num_limit=5, 
    method='tree', 
    positive='bad|1'
)

# 2. WOE转换
df_woe_orig = sc.woebin_ply(df, bins_original)
X_orig = df_woe_orig[[f'{f}_woe' for f in features]]
y = df['SeriousDlqin2yrs']

# 3. 划分训练集(70%)和验证集(30%)
X_train, X_val, y_train, y_val = train_test_split(X_orig, y, test_size=0.3, random_state=42)

# 4. 逻辑回归
lr = LogisticRegression(penalty=None)
lr.fit(X_train, y_train)

# 5. 验证集预测及指标
y_pred = lr.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, y_pred)
fpr, tpr, _ = roc_curve(y_val, y_pred)
ks = max(tpr - fpr)

print(f"原分箱验证集 AUC = {auc:.4f}")
print(f"原分箱验证集 KS = {ks:.4f}")

[INFO] creating woe binning ...
[INFO] converting into woe values ...
原分箱验证集 AUC = 0.8558
原分箱验证集 KS = 0.5581


In [52]:
import pandas as pd
import os

# 桌面路径
desktop = r"C:\Users\dp239\OneDrive\Desktop"

# 存放评分卡数据的列表
card_data = []

# 遍历评分卡字典
for var, item in card.items():
    if var == 'basepoints':
        # basepoints 是一个数值，转为 DataFrame
        tmp = pd.DataFrame({'variable': [var], 'bin': ['base'], 'points': [item]})
        card_data.append(tmp)
    else:
        # 检查 item 的类型
        if isinstance(item, pd.DataFrame):
            # 复制并添加 variable 列
            tmp = item.copy()
            tmp['variable'] = var
            # 确保有 bin 和 points 列（有时列名可能是 'bin' 和 'points'）
            # 如果列名不同，可自行调整，这里按常规写法
            if 'bin' in tmp.columns and 'points' in tmp.columns:
                card_data.append(tmp[['variable', 'bin', 'points']])
            else:
                # 如果列名不同，打印一下列名供调试
                print(f"变量 {var} 的列名: {tmp.columns.tolist()}")
                # 假设可能有 'bin' 和 'points' 的不同命名，例如 'bin' 和 'points' 存在则使用
                # 否则跳过或根据实际调整
                if 'bin' in tmp.columns and 'points' in tmp.columns:
                    card_data.append(tmp[['variable', 'bin', 'points']])
                else:
                    print(f"跳过变量 {var}，缺少 bin 或 points 列")
        else:
            print(f"变量 {var} 不是 DataFrame，类型为 {type(item)}，跳过")

# 合并所有 DataFrame
if card_data:
    card_df = pd.concat(card_data, ignore_index=True)
    # 保存到桌面
    output_path = os.path.join(desktop, 'scorecard.xlsx')
    card_df.to_excel(output_path, index=False)
    print(f"评分卡已导出到 {output_path}")
else:
    print("没有有效的评分卡数据可导出")

评分卡已导出到 C:\Users\dp239\OneDrive\Desktop\scorecard.xlsx
